# 学習済みモデルで英語の文法誤りを訂正する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/29_gec/gec_model_en.ipynb)

NLP クックブック 29-06 ／ 文法誤り訂正と文章校正

英語の文法誤り訂正を、公開されているファインチューニング済みモデルですぐに試したい。

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

name = "vennify/t5-base-grammar-correction"   # JFLEG で学習した T5
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name).eval()

def correct(sentences, num_beams=5):
    batch = tokenizer(["grammar: " + s for s in sentences], return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        out = model.generate(**batch, num_beams=num_beams, max_new_tokens=128)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

sentences = ["I has went to school yesterday and buyed a apple.", "She don't likes swimming in the see.",
             "Despite of the rain, we enjoyed.", "This sentence is already correct.", "He suggested me to go there."]
for s, c in zip(sentences, correct(sentences)):
    print(f"{s}\n  → {c}")

I has went to school yesterday and buyed a apple.
  → I went to school yesterday and bought an apple.
She don't likes swimming in the see.
  → She doesn't likes swimming in the see.
Despite of the rain, we enjoyed.
  → Despite the rain, we enjoyed.
This sentence is already correct.
  → This sentence is already correct.
He suggested me to go there.
  → He suggested I go there.


In [2]:
# 上位 3 つの訂正候補を見る
batch = tokenizer(["grammar: He suggested me to go there."], return_tensors="pt")
with torch.no_grad():
    out = model.generate(**batch, num_beams=5, num_return_sequences=3, max_new_tokens=64)
print(tokenizer.batch_decode(out, skip_special_tokens=True))

['He suggested I go there.', 'He suggested me to go there.', 'He suggested that I go there.']


- T5 は入力の先頭にタスクを表す接頭辞 (`grammar: `) を付ける約束で学習されている。付け忘れると訂正されない。
- 系列変換型は書き換えすぎる（正しい文まで変える）ことがある。逆に、直しきれないこともある。この出力では 2 文目が「She doesn't likes swimming in the see.」となり、likes と see の誤りが残った。編集タグを予測する方式 (GECToR) は高速で、過剰訂正が少ない。
- 現在は LLM による訂正が流暢性の面で強いが、必要以上に書き換える傾向がある。「最小限の訂正」を求める評価（CoNLL-2014、BEA-2019）では専用モデルが有利なことも多い。